# Find a dataset in your metadata cache

## Goal

Imagine you are looking for data about community facilities. Before downloading any
numbers, you want to know **what is available, what it describes and whether SSDU
can retrieve it**. In about 15 minutes, you will search a local catalogue, assess
two results and carry a useful dataset identifier into the provider workflow.

**Every catalogue entry in this lesson is invented.** No provider is contacted and
no observations are downloaded. The small result tables are enough to inspect directly.

## Setup

Read the results online, or follow [installation](https://xangma.github.io/ssdu/start/install.html)
to run the lesson in JupyterLab. Use the [lesson download links](https://xangma.github.io/ssdu/start/metadata-discovery.html)
to save this notebook and `metadata_discovery_fixture.py` into the **same folder**. Keep the companion's filename unchanged. It supplies two
invented responses to the real metadata-sync workflow using an in-memory HTTP
transport. You do not need to read its fixture code to follow the lesson.

Run cells in order with **Shift+Enter**. Files stay under `data/metadata-discovery`
beside your notebook. The setup creates a teaching catalogue; for your own work,
open a store populated by a [metadata capture](https://xangma.github.io/ssdu/guides/metadata-sync.html).

In [ ]:
from pathlib import Path
from textwrap import fill

import pandas as pd
from metadata_discovery_fixture import create_teaching_catalogue

from ssdu.metadata_index import MetadataIndex
from ssdu.metadata_store import MetadataStore

workspace = Path("data/metadata-discovery")
store_root = workspace / "catalogue"
teaching = create_teaching_catalogue(store_root)
index = MetadataIndex(store_root)
index.import_snapshot(teaching["snapshot_id"])
print("Ready: two invented descriptions, with their retained source responses.")

## Steps

### 1. Search the descriptions

`MetadataIndex` searches the metadata saved in this store. This is **local keyword
search**, not semantic search: it matches words in titles, descriptions and other
selected fields. Multiple words must all match. It makes no provider requests.

Use `limit` to keep the page small. `total` counts all matches; `next_offset` tells
you whether another page exists. Here, `community` should return two results.

In [ ]:
matches = index.search("community", limit=5)
summary = pd.DataFrame(
    {
        "provider": row["provider"],
        "title": row["title"],
        "SSDU retrieval": row["access"]["retrieval"]["status"],
    }
    for row in matches["items"]
)
print(f"Matches: {matches['total']}; next page: {matches['next_offset']}")
print(summary.to_string(index=False))

Both descriptions concern fictional communities. The MHCLG entry points to a CSV
that its connector can retrieve. The current Scottish portal entry is discoverable,
but its connector cannot acquire observations. Search results are **leads**, not
proof that the numbers answer your question.

This index differs from SSDU's dataset registry: the registry describes retained
datasets, while this index can describe datasets you have never downloaded.

### 2. Read one result carefully

Keep the full identifier, including the provider prefix, to avoid ambiguity. Narrow
by provider and ask for the detailed record.

In [ ]:
housing_match = index.search("community", provider="mhclg", limit=5)["items"][0]
selected = index.get(housing_match["dataset_id"])
print("Dataset:", selected["dataset_id"])
print("Description:")
print(fill(selected["description"], width=72))
print("Metadata source:", selected["access"]["metadata_url"])
print("Licence statement:", selected.get("licence") or "Not supplied")
print("Retrieval:", selected["access"]["retrieval"]["reason"])

The source link and invented licence statement remain attached to the result.
`access.retrieval.supported` describes SSDU's technical capability. It does **not**
grant permission to use or redistribute data. Read the provider's actual terms for
real work; `not_asserted` means access conditions have not been established.

### 3. Check freshness, coverage and completeness

A recently checked catalogue may still omit important detail. Compare the two
records before choosing one.

In [ ]:
quality = []
for row in matches["items"]:
    profile = row["metadata_profile"]
    quality.append(
        {
            "provider": row["provider"],
            "cache freshness": row["freshness"]["state"],
            "catalogue traversal": profile["completeness"]["status"],
            "description": row["field_status"]["description"]["status"],
            "time coverage": row["field_status"]["coverage.temporal"]["status"],
            "licence": row["field_status"]["licence"]["status"],
        }
    )
print(pd.DataFrame(quality).to_string(index=False))
print("Requested detail:", selected["metadata_profile"]["scope"]["detail"])
print("Page ceiling:", selected["metadata_profile"]["scope"]["max_pages"])
print("Last provider check:", selected["freshness"]["last_checked_at"])
print("Provider's stated modification:", selected["freshness"]["provider_modified_at"])

Immediately after setup, both cached records should be `fresh`: the default age
threshold is one day. Those check times concern our invented responses. The
provider's stated modification date is a separate clock. Offline reopening does
not update either check time.

`complete` means the **invented one-entry catalogue traversal** finished within its
bounds. It does not mean every metadata field is populated, every observation was
checked or every holding of the organisation is covered. Both records lack time
coverage; the Scottish record also lacks a licence statement. SSDU preserves these
gaps instead of inventing values.

The MHCLG spatial field says where a publication applies; it does not establish
the area codes actually present in its CSV. Inspect the retrieved table later.

### 4. Use the same cache from the command line

In a terminal opened beside this notebook, the equivalent search is:

```sh
ssdu --store data/metadata-discovery/catalogue metadata search community --limit 5
```

Use `metadata describe DATASET_ID` to read a result and `metadata status` to inspect
provider reports. These commands read local metadata. The next cell runs the search
using this notebook's Python installation and prints a compact view of its JSON.

In [ ]:
import json
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "ssdu.cli",
    "--store",
    str(store_root),
    "metadata",
    "search",
    "community",
    "--limit",
    "5",
]
cli_matches = json.loads(subprocess.check_output(command, text=True))
print(f"Command-line matches: {cli_matches['total']}")
for row in cli_matches["items"]:
    print(row["provider"], "—", row["title"])

### 5. Reopen a transferred catalogue offline

A metadata bundle contains the verified normalized descriptions and their retained
source responses. It contains **no observations** in this lesson. Export it, import
it into another store and search again. Here we also block SSDU HTTP calls so an
accidental request fails immediately.

In [ ]:
from unittest.mock import patch

from ssdu.http import HttpClient

metadata_store = MetadataStore(store_root)
bundle = workspace / "invented-metadata.zip"
if not bundle.exists():  # Reuse our teaching export when running the lesson again.
    metadata_store.export(teaching["snapshot_id"], bundle)
fresh_root = workspace / "reopened-catalogue"

with patch.object(HttpClient, "request", side_effect=AssertionError("HTTP is blocked")):
    fresh_store = MetadataStore(fresh_root)
    snapshot_id = fresh_store.import_bundle(bundle)
    fresh_store.verify(snapshot_id)
    reopened_index = MetadataIndex(fresh_root)
    reopened_index.import_snapshot(snapshot_id)
    replayed = reopened_index.search("community", limit=5)

print("Offline matches:", replayed["total"])
print("Original snapshot retained:", snapshot_id == teaching["snapshot_id"])

## Checks

We should have the same two dataset identifiers after transfer, with different
retrieval capabilities. The assertions below check those outcomes. No download
of the CSV has taken place.

In [ ]:
assert snapshot_id == teaching["snapshot_id"]
assert matches["total"] == cli_matches["total"] == replayed["total"] == 2
assert {row["dataset_id"] for row in matches["items"]} == {
    row["dataset_id"] for row in replayed["items"]
}
assert {row["dataset_id"]: row["freshness"]["last_checked_at"] for row in matches["items"]} == {
    row["dataset_id"]: row["freshness"]["last_checked_at"] for row in replayed["items"]
}
assert selected["access"]["retrieval"]["supported"] is True
scottish = index.search("community", provider="scotland")["items"][0]
assert scottish["access"]["retrieval"]["supported"] is False
assert not any((store_root / "snapshots").iterdir())
print("Checks passed: metadata searched and replayed; no observations retained.")

## Next Steps

For a real catalogue, keep the chosen record's `provider` and `native_id`. Use them
to open a `Connector`, inspect its description and available selections, then
review a bounded retrieval plan. The
[provider-selection tutorial](https://xangma.github.io/ssdu/start/provider-selection.html)
walks through that step with a separate invented dataset.

The two invented records here cannot be looked up on the real services. To capture
your own catalogue, follow
[Keep a catalogue you can search offline](https://xangma.github.io/ssdu/guides/metadata-sync.html).
The [coverage matrix](https://xangma.github.io/ssdu/reference/sources.html) explains
which services support observation retrieval.